# 🛡️ Clasificador de Vulnerabilidades en Código Fuente
## Dataset: CVEFixes — Modelo de Machine Learning Tradicional

**Objetivo:** Clasificar fragmentos de código fuente como `SEGURO (0)` o `VULNERABLE (1)`  
**Restricción estricta:** Solo librerías clásicas de ML (Scikit-Learn, XGBoost). Prohibido DL/LLMs.  
**Meta:** ≥ 82% Accuracy en validación cruzada de 5 pliegues.

---

## 📦 Instalación de Dependencias

In [21]:
# Instalar dependencias necesarias (ejecutar solo si no están instaladas)
!pip install xgboost imbalanced-learn scikit-learn pandas numpy joblib matplotlib

  Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 4.5 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (363 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.9 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl 

---
## 📁 BLOQUE 1: Carga, Análisis y Balanceo Avanzado del Dataset

Se simula la carga del dataset CVEFixes con ejemplos representativos de código  
**seguro** e **inseguro**, cubriendo patrones reales de vulnerabilidades (SQLi, XSS,  
command injection, path traversal, etc.).

In [40]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

# =============================================================================
# 📁 BLOQUE 1: Carga, Limpieza y Balanceo Dinámico del Dataset Real CVEFixes
# =============================================================================

PATH_DATASET = 'CVEFixes.csv'
COLUMNA_CODIGO = 'code'         
COLUMNA_ETIQUETA = 'safety'  

if not os.path.exists(PATH_DATASET):
    raise FileNotFoundError(
        f"❌ No se encontró el archivo '{PATH_DATASET}'. "
        f"Asegúrate de que esté en la raíz de tu proyecto."
    )

print("⏳ Leyendo el dataset real e industrial de CVEFixes...")

# 1. Carga optimizada de las columnas necesarias
df_raw = pd.read_csv(PATH_DATASET, usecols=[COLUMNA_CODIGO, COLUMNA_ETIQUETA])

# 2. Limpieza de datos nulos o registros corruptos
df_raw = df_raw.dropna(subset=[COLUMNA_CODIGO, COLUMNA_ETIQUETA]).reset_index(drop=True)
df_raw[COLUMNA_CODIGO] = df_raw[COLUMNA_CODIGO].astype(str)
df_raw[COLUMNA_ETIQUETA] = df_raw[COLUMNA_ETIQUETA].astype(str).str.strip().str.lower()

# 🔄 MAPEADO HOMOLOGADO (Texto a Binario de producción)
# 'vulnerable' se convierte en 1 | 'safe' (o cualquier otro) se convierte en 0
df_raw['is_vulnerable'] = df_raw[COLUMNA_ETIQUETA].apply(lambda x: 1 if 'vulnerable' in x else 0)

# Renombrar columna de código para compatibilidad total con el Bloque 2 (AST/Regex)
df_raw = df_raw.rename(columns={COLUMNA_CODIGO: 'code_content'})

print(f"✅ Archivo mapeado con éxito. Total de registros crudos: {len(df_raw)}")

# 3. Separación analítica de las clases para balanceo estratégico
df_vulnerables = df_raw[df_raw['is_vulnerable'] == 1]
df_seguros = df_raw[df_raw['is_vulnerable'] == 0]

n_vulnerables = len(df_vulnerables)
n_seguros = len(df_seguros)

print(f"   ├─ Muestras Vulnerables (Clase 1): {n_vulnerables}")
print(f"   └─ Muestras Seguras (Clase 0):     {n_seguros}")

# 4. BALANCEO AUTOMÁTICO DE CLASES (Rúbrica de Minería de Datos)
n_muestras_balance = min(n_vulnerables, n_seguros)

# Límite seguro para cuidar la memoria RAM y el venv de tu máquina
# Procesar miles de filas multilenguaje toma su tiempo, 4000 por clase es un estándar robusto.
LIMITE_SEGURO_RAM = 4000 
muestras_por_clase = min(n_muestras_balance, LIMITE_SEGURO_RAM)

print(f"⚖️ Ajustando dataset balanceado a {muestras_por_clase} muestras por clase.")

df_vuln_bal = df_vulnerables.sample(n=muestras_por_clase, random_state=42)
df_safe_bal = df_seguros.sample(n=muestras_por_clase, random_state=42)

# Combinamos y barajamos el set final
df = pd.concat([df_vuln_bal, df_safe_bal], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Limpiar columnas intermedias redundantes para no arrastrar basura en el DataFrame
if COLUMNA_ETIQUETA in df.columns:
    df = df.drop(columns=[COLUMNA_ETIQUETA])

# =============================================================================
# 📊 ANÁLISIS EXPLORATORIO DEL DATASET INDUSTRIAL REAL
# =============================================================================
print('\n' + '=' * 60)
print('📊 ANÁLISIS EXPLORATORIO DEL DATASET REAL FINAL')
print('=' * 60)
print(f'🔢 Total de muestras balanceadas listas para el pipeline: {len(df)}')
print(f'\n📈 Distribución simétrica de clases:')
print(df['is_vulnerable'].value_counts())
print('=' * 60 + '\n')

⏳ Leyendo el dataset real e industrial de CVEFixes...
✅ Archivo mapeado con éxito. Total de registros crudos: 31160
   ├─ Muestras Vulnerables (Clase 1): 15592
   └─ Muestras Seguras (Clase 0):     15568
⚖️ Ajustando dataset balanceado a 4000 muestras por clase.

📊 ANÁLISIS EXPLORATORIO DEL DATASET REAL FINAL
🔢 Total de muestras balanceadas listas para el pipeline: 8000

📈 Distribución simétrica de clases:
is_vulnerable
1    4000
0    4000
Name: count, dtype: int64



---
## 🔬 BLOQUE 2: Pipeline de Extracción Extensible (AST + Regex Avanzado)

Se extraen **14 features estructurales** del código fuente:
- Métricas AST (profundidad, nodos, complejidad ciclomática estimada)
- Llamadas a funciones peligrosas (eval, exec, subprocess, SQL raw)
- Presencia de mecanismos de sanitización
- Métricas de entropía y estructura del código

In [ ]:
import ast
import re
import math

# =============================================================================
# FUNCIÓN AUXILIAR: Profundidad máxima del AST
# =============================================================================
def ast_depth(tree):
    """Calcula recursivamente la profundidad máxima del árbol AST."""
    if not isinstance(tree, ast.AST):
        return 0
    children = list(ast.iter_child_nodes(tree))
    if not children:
        return 1
    return 1 + max(ast_depth(child) for child in children)

# =============================================================================
# FUNCIÓN AUXILIAR: Entropía de Shannon del código
# Mide la aleatoriedad/ofuscación — código obfuscado tiende a tener alta entropía
# =============================================================================
def shannon_entropy(text):
    """Calcula la entropía de Shannon de una cadena de texto."""
    if not text:
        return 0.0
    freq = {}
    for c in text:
        freq[c] = freq.get(c, 0) + 1
    n = len(text)
    return -sum((f/n) * math.log2(f/n) for f in freq.values())

# =============================================================================
# PATRONES REGEX AVANZADOS PARA DETECCIÓN DE VULNERABILIDADES
# Diseñados para capturar patrones reales de CVEs sin perder operadores
# =============================================================================

# Funciones intrínsecamente peligrosas
PATRON_FUNCIONES_PELIGROSAS = re.compile(
    r'\b(eval|exec|execfile|compile|__import__|getattr|setattr|delattr)\s*\(',
    re.IGNORECASE
)

# Subprocess con shell=True (vector de command injection)
PATRON_SUBPROCESS_SHELL = re.compile(
    r'subprocess\.(call|run|Popen|check_output|check_call)\s*\([^)]*shell\s*=\s*True',
    re.IGNORECASE | re.DOTALL
)

# Subprocess genérico (cualquier uso)
PATRON_SUBPROCESS_GENERICO = re.compile(
    r'subprocess\.(call|run|Popen|check_output|check_call|getoutput)\s*\(',
    re.IGNORECASE
)

# os.system / os.popen — otra superficie de command injection
PATRON_OS_COMMANDS = re.compile(
    r'os\.(system|popen|execv|execve|execvp|execvpe|spawnv|spawnve)\s*\(',
    re.IGNORECASE
)

# SQL raw: concatenación o formateo de strings en queries
PATRON_SQL_CRUDO = re.compile(
    r'(\.execute\s*\([^?%]*(\+|%|format|f[\'"]|\{)|"\'\s*\+\s*\w+\s*\+\s*"\')',
    re.IGNORECASE
)

# SQL con f-strings o % formatting (inyección directa)
PATRON_SQL_FSTRING = re.compile(
    r'(SELECT|INSERT|UPDATE|DELETE|DROP|UNION|FROM|WHERE)[^;]*"\'\s*(%|format|f\'|f\")',
    re.IGNORECASE
)

# Serialización peligrosa
PATRON_PICKLE = re.compile(
    r'\bpickle\.(loads|load|Unpickler)\s*\(',
    re.IGNORECASE
)

# Concatenación de strings en rutas de archivos (path traversal)
PATRON_PATH_CONCAT = re.compile(
    r'(open|file)\s*\(\s*[\w\'"]+\s*\+\s*\w+',
    re.IGNORECASE
)

# Hashing débil (MD5, SHA1 para contraseñas)
PATRON_HASH_DEBIL = re.compile(
    r'hashlib\.(md5|sha1)\s*\(',
    re.IGNORECASE
)

# Uso de random() en contexto de seguridad
PATRON_RANDOM_INSEGURO = re.compile(
    r'random\.(randint|random|choice|shuffle|sample)\s*\(',
    re.IGNORECASE
)

# --- Patrones de SANITIZACIÓN (código seguro) ---
PATRON_SANITIZACION = re.compile(
    r'\b(escape|sanitize|sanitise|is_valid|validate|clean|purify|'
    r'strip_tags|bleach|html\.escape|markupsafe|'
    r'parameterize|prepared_statement|placeholder|'
    r'allowed_extensions|whitelist|ALLOWED|safe_path|realpath|'
    r'compare_digest|escape_filter_chars|defusedxml|'
    r'safe_load|create_default_context|token_urlsafe|secrets\.)\b',
    re.IGNORECASE
)

# SQL parameterizado (seguro)
PATRON_SQL_PARAMETERIZADO = re.compile(
    r'\.execute\s*\(\s*[\'\"][^\'\"]*(\?|%s|:param|:\w+)[\'\"]',
    re.IGNORECASE
)

# Variables de entorno para credenciales (seguro)
PATRON_ENV_VARS = re.compile(
    r'os\.environ\.(get|__getitem__)\s*\(',
    re.IGNORECASE
)

# Validación explícita de tipos (señal de código defensivo)
PATRON_VALIDACION_TIPOS = re.compile(
    r'isinstance\s*\(|type\s*\(\w+\)\s*==|assert\s+',
    re.IGNORECASE
)

# Manejo de excepciones (puede indicar código robusto)
PATRON_EXCEPCIONES = re.compile(
    r'\btry\s*:.*?\bexcept\b',
    re.IGNORECASE | re.DOTALL
)

# Imports de librerías de seguridad
PATRON_IMPORTS_SEGUROS = re.compile(
    r'import\s+(bcrypt|argon2|cryptography|secrets|hmac|defusedxml|bleach)',
    re.IGNORECASE
)

# =============================================================================
# FUNCIÓN PRINCIPAL: extraer_features_codigo(codigo)
# =============================================================================
def extraer_features_codigo(codigo):
    """
    Extrae un vector de features estructurales y semánticos del código fuente.
    
    Retorna un diccionario con las siguientes features:
    - ast_depth: Profundidad máxima del AST (feature AST obligatoria)
    - ast_node_count: Total de nodos en el AST
    - ast_parse_error: 1 si el código no parsea correctamente (señal de ofuscación)
    - func_calls_count: Número total de llamadas a funciones
    - dangerous_func_calls: Conteo de eval/exec/getattr peligrosos
    - subprocess_shell: Uso de subprocess con shell=True
    - subprocess_any: Cualquier uso de subprocess
    - os_commands: Uso de os.system/popen
    - sql_raw_concat: Concatenación directa en queries SQL
    - pickle_usage: Uso de deserialización pickle
    - path_concat: Concatenación en rutas de archivos
    - weak_hash: Uso de MD5/SHA1 para passwords
    - insecure_random: Uso de random() en contexto de seguridad
    - sanitization_present: Presencia de funciones de sanitización
    - sql_parameterized: SQL con parámetros seguros (?/%s)
    - env_vars_used: Uso de variables de entorno para credenciales
    - type_validation: Validación explícita de tipos
    - exception_handling: Manejo de excepciones presente
    - secure_imports: Imports de librerías de seguridad
    - shannon_entropy: Entropía de Shannon (código ofuscado = alta entropía)
    - lines_count: Número de líneas de código
    - string_concat_count: Número de concatenaciones de strings con +
    - total_danger_score: Score compuesto de peligrosidad
    - total_safety_score: Score compuesto de seguridad
    """
    features = {}
    
    # -------------------------------------------------------------------------
    # Feature 1-3: Análisis AST (con manejo de errores de sintaxis)
    # -------------------------------------------------------------------------
    try:
        tree = ast.parse(codigo)
        features['ast_depth'] = ast_depth(tree)          # OBLIGATORIA: profundidad AST
        features['ast_node_count'] = sum(1 for _ in ast.walk(tree))  # total nodos
        features['ast_parse_error'] = 0
        
        # Conteo de llamadas a funciones vía AST (más preciso que regex)
        features['func_calls_count'] = sum(
            1 for node in ast.walk(tree) if isinstance(node, ast.Call)
        )
        
        # Estimación de complejidad ciclomática (if, for, while, try, with)
        branch_nodes = (ast.If, ast.For, ast.While, ast.Try, ast.ExceptHandler,
                       ast.With, ast.Assert, ast.comprehension)
        features['cyclomatic_complexity'] = sum(
            1 for node in ast.walk(tree) if isinstance(node, branch_nodes)
        ) + 1
        
    except SyntaxError:
        # Código que no parsea: puede ser multilenguaje, obfuscado o fragmento
        features['ast_depth'] = 0
        features['ast_node_count'] = 0
        features['ast_parse_error'] = 1
        features['func_calls_count'] = 0
        features['cyclomatic_complexity'] = 1
    
    # -------------------------------------------------------------------------
    # Feature 4-12: Llamadas a funciones peligrosas (OBLIGATORIAS en rúbrica)
    # -------------------------------------------------------------------------
    features['dangerous_func_calls'] = len(PATRON_FUNCIONES_PELIGROSAS.findall(codigo))  # OBLIGATORIA
    features['subprocess_shell'] = 1 if PATRON_SUBPROCESS_SHELL.search(codigo) else 0    # OBLIGATORIA
    features['subprocess_any'] = 1 if PATRON_SUBPROCESS_GENERICO.search(codigo) else 0   # OBLIGATORIA
    features['os_commands'] = 1 if PATRON_OS_COMMANDS.search(codigo) else 0
    features['sql_raw_concat'] = 1 if PATRON_SQL_CRUDO.search(codigo) else 0             # OBLIGATORIA (SQL raw)
    features['sql_fstring'] = 1 if PATRON_SQL_FSTRING.search(codigo) else 0
    features['pickle_usage'] = 1 if PATRON_PICKLE.search(codigo) else 0
    features['path_concat'] = 1 if PATRON_PATH_CONCAT.search(codigo) else 0
    features['weak_hash'] = 1 if PATRON_HASH_DEBIL.search(codigo) else 0
    features['insecure_random'] = 1 if PATRON_RANDOM_INSEGURO.search(codigo) else 0
    
    # -------------------------------------------------------------------------
    # Feature 13-17: Presencia de sanitización/escapes (OBLIGATORIAS en rúbrica)
    # -------------------------------------------------------------------------
    features['sanitization_present'] = 1 if PATRON_SANITIZACION.search(codigo) else 0    # OBLIGATORIA
    features['sql_parameterized'] = 1 if PATRON_SQL_PARAMETERIZADO.search(codigo) else 0 # OBLIGATORIA
    features['env_vars_used'] = 1 if PATRON_ENV_VARS.search(codigo) else 0
    features['type_validation'] = 1 if PATRON_VALIDACION_TIPOS.search(codigo) else 0
    features['exception_handling'] = 1 if PATRON_EXCEPCIONES.search(codigo) else 0
    features['secure_imports'] = 1 if PATRON_IMPORTS_SEGUROS.search(codigo) else 0
    
    # -------------------------------------------------------------------------
    # Feature 18-22: Métricas estructurales avanzadas
    # -------------------------------------------------------------------------
    features['shannon_entropy'] = shannon_entropy(codigo)
    features['lines_count'] = codigo.count('\n') + 1
    
    # Conteo de concatenaciones de strings con '+' (indicador de construcción dinámica)
    features['string_concat_count'] = len(re.findall(r"['\"][^'\"]*['\"]\s*\+|\+\s*['\"][^'\"]*['\"]", codigo))
    
    # Presencia de comentarios de seguridad (código maduro suele tener anotaciones)
    features['security_comments'] = len(re.findall(
        r'#.*(segur|safe|valid|sanitiz|escape|authen|authori|permis)',
        codigo, re.IGNORECASE
    ))
    
    # Uso de raise (indicador de validación defensiva)
    features['raise_count'] = len(re.findall(r'\braise\b', codigo))
    
    # -------------------------------------------------------------------------
    # Feature 23-24: Scores compuestos (ponderación experta de señales)
    # -------------------------------------------------------------------------
    features['total_danger_score'] = (
        features['dangerous_func_calls'] * 3 +
        features['subprocess_shell'] * 3 +
        features['subprocess_any'] * 1 +
        features['os_commands'] * 2 +
        features['sql_raw_concat'] * 3 +
        features['sql_fstring'] * 3 +
        features['pickle_usage'] * 2 +
        features['path_concat'] * 2 +
        features['weak_hash'] * 1 +
        features['insecure_random'] * 1 +
        features['string_concat_count'] * 1
    )
    
    features['total_safety_score'] = (
        features['sanitization_present'] * 3 +
        features['sql_parameterized'] * 3 +
        features['env_vars_used'] * 2 +
        features['type_validation'] * 2 +
        features['exception_handling'] * 1 +
        features['secure_imports'] * 3 +
        features['security_comments'] * 1 +
        features['raise_count'] * 1
    )
    
    return features


# =============================================================================
# APLICAR EXTRACCIÓN DE FEATURES AL DATASET COMPLETO
# =============================================================================
print('🔬 Extrayendo features estructurales del código fuente...')
features_list = df['code_content'].apply(extraer_features_codigo)
df_features = pd.DataFrame(features_list.tolist())

print(f'✅ Features extraídas: {len(df_features.columns)} variables estructurales')
print(f'\n📋 Estadísticas de features por clase:')
df_analisis = pd.concat([df_features, df['is_vulnerable']], axis=1)
print(df_analisis.groupby('is_vulnerable')[['total_danger_score', 'total_safety_score', 
                                             'ast_depth', 'dangerous_func_calls']].mean().round(3))

# Mostrar correlación de features con la etiqueta
print('\n📊 Top 10 Features más correlacionadas con vulnerabilidad:')
correlaciones = df_analisis.corr()['is_vulnerable'].abs().sort_values(ascending=False)
print(correlaciones.head(11).to_string())

---
## 🔤 BLOQUE 3: Vectorización Personalizada para Código Fuente

Se configura un `TfidfVectorizer` **especializado para código**, con un `token_pattern`
que captura tanto palabras clave como caracteres sintácticos críticos (`+`, `=`, `(`, `[`, etc.)
usando n-grams para detectar patrones multi-token.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix, hstack
import numpy as np

# =============================================================================
# PREPARACIÓN DE DATOS PARA VECTORIZACIÓN
# =============================================================================
X_text = df['code_content']
X_struct = df_features.values  # Matriz de features estructurales
y = df['is_vulnerable'].values

# =============================================================================
# TFIDF VECTORIZER EXPERTO PARA CÓDIGO FUENTE
#
# token_pattern: Captura:
#   - Identificadores Python/C++: [a-zA-Z_][\w]* (incluyendo snake_case)
#   - Números y literales: \d+
#   - Operadores críticos: ==, !=, <=, >=, +=, -=, ->, :=
#   - Caracteres sintácticos individuales: ()[]{}=+-*/%<>!&|^~@
#   - Strings y comillas
# =============================================================================
TOKEN_PATTERN_CODIGO = (
    r'(?:'                          # Inicio de grupo no capturante
    r'[a-zA-Z_][\w]*'              # Identificadores (funciones, variables)
    r'|\d+\.?\d*'                  # Números enteros y decimales
    r'|==|!=|<=|>=|\+=|-=|\*=|/=|->|:='  # Operadores compuestos críticos
    r'|[()\[\]{}=+\-*/%<>!&|^~@,.:;]'    # Operadores y delimitadores individuales
    r')'
)

vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN_CODIGO,
    ngram_range=(1, 3),          # Unigrams, bigrams y trigrams para capturar patrones multi-token
    min_df=2,                    # Ignorar tokens que aparecen en menos de 2 documentos
    max_df=0.95,                 # Ignorar tokens demasiado frecuentes (como 'def', ':')
    max_features=8000,           # Límite de vocabulary para controlar dimensionalidad
    sublinear_tf=True,           # Aplicar log(1+tf): reduce efecto de frecuencias muy altas
    strip_accents='unicode',     # Normalizar caracteres unicode
    analyzer='word',
)

# Fit y transformación de la matrix TF-IDF
X_tfidf = vectorizer.fit_transform(X_text)

print('=' * 60)
print('🔤 VECTORIZACIÓN TF-IDF PARA CÓDIGO FUENTE')
print('=' * 60)
print(f'📐 Dimensiones matriz TF-IDF: {X_tfidf.shape}')
print(f'📊 Sparsity: {1.0 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.4f}')
print(f'\n🔑 Muestra de tokens capturados (primeros 30):')
tokens_vocab = vectorizer.get_feature_names_out()
print(list(tokens_vocab[:30]))

# Tokens más discriminativos para código vulnerable
print(f'\n⚠️  Tokens más asociados a código VULNERABLE (mayor TF-IDF promedio):')
vuln_idx = np.where(y == 1)[0]
safe_idx = np.where(y == 0)[0]
tfidf_vuln_mean = np.asarray(X_tfidf[vuln_idx].mean(axis=0)).flatten()
tfidf_safe_mean = np.asarray(X_tfidf[safe_idx].mean(axis=0)).flatten()
diff_scores = tfidf_vuln_mean - tfidf_safe_mean
top_vuln_tokens_idx = np.argsort(diff_scores)[::-1][:15]
print([tokens_vocab[i] for i in top_vuln_tokens_idx])

print(f'\n✅ Tokens más asociados a código SEGURO:')
top_safe_tokens_idx = np.argsort(diff_scores)[:15]
print([tokens_vocab[i] for i in top_safe_tokens_idx])

---
## 🔀 BLOQUE 4: Fusión de Características y Clasificador Tradicional de Alto Rendimiento

Se combina la matriz TF-IDF con las features estructurales en una **pipeline unificada**,
aplicando SMOTE para el balanceo y Random Forest + XGBoost como clasificadores.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.sparse import csr_matrix, hstack, issparse
import xgboost as xgb

# =============================================================================
# FUSIÓN DE FEATURES: TF-IDF + Features Estructurales
# Estrategia: Convertir features estructurales a sparse matrix y concatenar
# horizontalmente con la matriz TF-IDF para una sola matriz X unificada
# =============================================================================

# Normalizar features estructurales a [0, 1] para equiparar escala con TF-IDF
scaler = MinMaxScaler()
X_struct_scaled = scaler.fit_transform(X_struct.astype(float))
X_struct_sparse = csr_matrix(X_struct_scaled)

# Concatenación horizontal: [TF-IDF | Features Estructurales]
X_combined = hstack([X_tfidf, X_struct_sparse])

print('=' * 60)
print('🔀 FUSIÓN DE CARACTERÍSTICAS')
print('=' * 60)
print(f'📐 Dimensiones finales de X_combined: {X_combined.shape}')
print(f'   - TF-IDF features: {X_tfidf.shape[1]}')
print(f'   - Features estructurales: {X_struct_sparse.shape[1]}')

# =============================================================================
# SELECCIÓN DE FEATURES: SelectKBest con F-score
# Reduce dimensionalidad manteniendo las features más discriminativas
# =============================================================================
# Nota: chi2 requiere valores no negativos (TF-IDF y MinMaxScaler garantizan esto)
selector = SelectKBest(score_func=chi2, k=3000)
X_selected = selector.fit_transform(X_combined, y)

print(f'\n🎯 Features seleccionadas por SelectKBest(chi2, k=3000): {X_selected.shape[1]}')
print(f'   Reducción: {X_combined.shape[1]} → {X_selected.shape[1]} features')

# =============================================================================
# BALANCEO CON SMOTE (Synthetic Minority Oversampling Technique)
# Genera muestras sintéticas de la clase minoritaria interpolando en el espacio
# de features, evitando el sesgo por desbalance de clases
# =============================================================================
smote = SMOTE(random_state=42, k_neighbors=5)
X_bal, y_bal = smote.fit_resample(X_selected, y)

print(f'\n⚖️  BALANCEO CON SMOTE:')
print(f'   Antes: {np.bincount(y)} (0=Seguro, 1=Vulnerable)')
print(f'   Después: {np.bincount(y_bal)} (0=Seguro, 1=Vulnerable)')
print(f'   Total muestras para entrenamiento: {X_bal.shape[0]}')
# =============================================================================
# DEFINICIÓN DE CLASIFICADORES DE ALTO RENDIMIENTO
# 
# Estrategia: Ensemble de Random Forest + XGBoost con VotingClassifier
# - Random Forest: robusto, maneja bien alta dimensionalidad, resistente a overfitting
# - XGBoost: captura interacciones no lineales complejas entre features
# - Voting 'soft': promedia probabilidades para decisión más suave y precisa
# =============================================================================

# Random Forest optimizado para clasificación de texto de código
rf_clf = RandomForestClassifier(
    n_estimators=300,          # 300 árboles: buen balance entre rendimiento y tiempo
    max_depth=20,              # Profundidad máxima para capturar patrones complejos
    min_samples_split=5,       # Evitar divisiones con muy pocas muestras (anti-overfitting)
    min_samples_leaf=2,        # Mínimo de muestras en hojas terminales
    max_features='sqrt',       # Sqrt de features por árbol: estándar para clasificación
    class_weight='balanced',   # Peso inverso a frecuencia de clase (redundante con SMOTE, pero refuerza)
    random_state=42,
    n_jobs=-1,                 # Paralelizar en todos los núcleos
    oob_score=True,            # Out-of-bag score como estimación interna del error
)

# XGBoost optimizado: gradient boosting con regularización L1+L2
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,          # 300 boosting rounds
    max_depth=6,               # Árboles más superficiales para generalización
    learning_rate=0.1,         # Tasa de aprendizaje moderada
    subsample=0.8,             # 80% de muestras por árbol (bagging)
    colsample_bytree=0.8,      # 80% de features por árbol
    reg_alpha=0.1,             # Regularización L1 para sparse features
    reg_lambda=1.0,            # Regularización L2 (weight decay)
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',        # Método histograma: más rápido para datasets grandes
)

# Ensemble VotingClassifier (soft voting = promedio de probabilidades)
ensemble_clf = VotingClassifier(
    estimators=[
        ('random_forest', rf_clf),
        ('xgboost', xgb_clf)
    ],
    voting='soft',             # Soft voting para mayor precisión
    weights=[1, 1],            # Mismo peso a ambos clasificadores
    n_jobs=-1
)

print('\n🤖 CLASIFICADORES DEFINIDOS:')
print('   1. RandomForestClassifier (n_estimators=300, max_depth=20)')
print('   2. XGBClassifier (n_estimators=300, max_depth=6, lr=0.1)')
print('   3. VotingClassifier (Ensemble soft de RF + XGB) ← MODELO PRINCIPAL')
print('\n✅ Dataset listo para validación cruzada')

---
## 📊 BLOQUE 5: Validación Cruzada e Impresión de Evidencias (Meta: ≥ 82%)

Se aplica **5-Fold Cross-Validation** sobre el dataset balanceado con SMOTE,
evaluando múltiples métricas: Accuracy, F1, Precision, Recall y AUC-ROC.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix
import time

# =============================================================================
# CONFIGURACIÓN DE VALIDACIÓN CRUZADA ESTRATIFICADA
# StratifiedKFold asegura que cada pliegue mantenga la proporción de clases
# =============================================================================
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =============================================================================
# EVALUACIÓN DE TODOS LOS CLASIFICADORES PARA COMPARACIÓN
# =============================================================================
print('=' * 70)
print('📊 VALIDACIÓN CRUZADA DE 5 PLIEGUES — EVIDENCIA DE RENDIMIENTO')
print('=' * 70)
print(f'   Dataset: {X_bal.shape[0]} muestras × {X_bal.shape[1]} features')
print(f'   Estrategia: StratifiedKFold(n_splits=5, shuffle=True)')
print()

resultados = {}

for nombre, clf in [
    ('Random Forest', rf_clf),
    ('XGBoost', xgb_clf),
    ('Ensemble RF+XGB', ensemble_clf)
]:
    print(f'⏳ Evaluando: {nombre}...')
    t0 = time.time()
    
    # Múltiples métricas en una sola llamada
    cv_results = cross_validate(
        clf, X_bal, y_bal,
        cv=cv_strategy,
        scoring={
            'accuracy': 'accuracy',
            'f1': 'f1',
            'precision': 'precision',
            'recall': 'recall',
            'roc_auc': 'roc_auc'
        },
        n_jobs=-1
    )
    
    elapsed = time.time() - t0
    resultados[nombre] = cv_results
    
    acc_mean = cv_results['test_accuracy'].mean()
    acc_std = cv_results['test_accuracy'].std()
    f1_mean = cv_results['test_f1'].mean()
    prec_mean = cv_results['test_precision'].mean()
    rec_mean = cv_results['test_recall'].mean()
    auc_mean = cv_results['test_roc_auc'].mean()
    
    estado = '✅ APRUEBA' if acc_mean >= 0.82 else '❌ NO APRUEBA'
    
    print(f'\n  ┌─ {nombre} {estado} ({elapsed:.1f}s)')
    print(f'  │  Accuracy:  {acc_mean:.4f} ± {acc_std:.4f}  (Por pliegue: {[f"{v:.4f}" for v in cv_results["test_accuracy"]]})')
    print(f'  │  F1-Score:  {f1_mean:.4f}')
    print(f'  │  Precision: {prec_mean:.4f}')
    print(f'  │  Recall:    {rec_mean:.4f}')
    print(f'  └─ AUC-ROC:  {auc_mean:.4f}')
    print()

# =============================================================================
# RESUMEN EJECUTIVO
# =============================================================================
print('\n' + '=' * 70)
print('🏆 RESUMEN EJECUTIVO — COMPARACIÓN DE MODELOS')
print('=' * 70)
print(f'{"Modelo":<20} {"Accuracy":<12} {"F1":<10} {"AUC-ROC":<10} {"Estado"}')
print('-' * 70)
for nombre, res in resultados.items():
    acc = res['test_accuracy'].mean()
    f1 = res['test_f1'].mean()
    auc = res['test_roc_auc'].mean()
    estado = '✅ APRUEBA' if acc >= 0.82 else '❌ NO APRUEBA'
    print(f'{nombre:<20} {acc:.4f}       {f1:.4f}     {auc:.4f}     {estado}')

print('\n📌 UMBRAL REQUERIDO: 82% Accuracy (5-Fold Cross-Validation)')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# =============================================================================
# EVALUACIÓN DETALLADA DEL MODELO FINAL (Ensemble RF+XGB)
# Train/Test split para reporte de clasificación y matriz de confusión
# =============================================================================
print('=' * 70)
print('🔍 REPORTE DETALLADO — MODELO FINAL: Ensemble RF+XGB')
print('=' * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

# Entrenar el modelo ensemble final
print('\n⏳ Entrenando modelo final sobre 80% de los datos...')
ensemble_clf.fit(X_train, y_train)
y_pred = ensemble_clf.predict(X_test)

print('\n📋 REPORTE DE CLASIFICACIÓN:')
print(classification_report(
    y_test, y_pred,
    target_names=['SEGURO (0)', 'VULNERABLE (1)'],
    digits=4
))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print('🔢 MATRIZ DE CONFUSIÓN:')
print('   Predicho:  SEGURO   VULNERABLE')
print(f'   Real SEGURO:     {cm[0][0]:>5}    {cm[0][1]:>5}')
print(f'   Real VULN:       {cm[1][0]:>5}    {cm[1][1]:>5}')
print(f'\n   Falsos Positivos (código seguro clasificado como vulnerable): {cm[0][1]}')
print(f'   Falsos Negativos (vulnerabilidad no detectada): {cm[1][0]}')
print('   → En ciberseguridad, minimizar FN es crítico (missed vulnerabilities)')

# -----------------------------------------------------------------------
# OOB Score del Random Forest como validación adicional
# -----------------------------------------------------------------------
print('\n📌 OOB Score (Random Forest): Estimación interna de error generalizacion')
rf_standalone = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1, oob_score=True
)
rf_standalone.fit(X_bal, y_bal)
print(f'   Random Forest OOB Accuracy: {rf_standalone.oob_score_:.4f}')

---
## 💾 BLOQUE 6: Exportación de Artefactos de Inferencia

Se exportan todos los componentes necesarios para la inferencia en producción:
- Modelo ensemble entrenado
- Vectorizador TF-IDF
- Selector de features
- Scaler de features estructurales
- Metadata del modelo

In [ ]:
import joblib
import os
import json
from datetime import datetime

# =============================================================================
# ENTRENAMIENTO FINAL DEL MODELO SOBRE TODOS LOS DATOS BALANCEADOS
# Después de la validación cruzada, entrenamos sobre el dataset completo
# para maximizar la información aprendida antes de exportar
# =============================================================================
print('=' * 60)
print('💾 EXPORTACIÓN DE ARTEFACTOS DE INFERENCIA')
print('=' * 60)

print('\n⏳ Entrenando modelo final sobre 100% de datos balanceados...')
ensemble_clf.fit(X_bal, y_bal)
print('✅ Entrenamiento completado')

# Crear directorio de artefactos
artifacts_dir = 'model_artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

# =============================================================================
# EXPORTACIÓN DE ARTEFACTOS CON JOBLIB (compresión nivel 3)
# =============================================================================
artefactos = {
    'modelo_ensemble': ensemble_clf,       # Modelo principal (RF + XGB voting)
    'vectorizador_tfidf': vectorizer,      # TF-IDF vectorizer ajustado
    'selector_features': selector,         # SelectKBest(chi2, k=3000)
    'scaler_estructural': scaler,          # MinMaxScaler para features AST/regex
}

rutas_exportadas = {}
for nombre, artefacto in artefactos.items():
    ruta = os.path.join(artifacts_dir, f'{nombre}.joblib')
    joblib.dump(artefacto, ruta, compress=3)
    size_kb = os.path.getsize(ruta) / 1024
    rutas_exportadas[nombre] = ruta
    print(f'  📦 {nombre}.joblib → {size_kb:.1f} KB')

# =============================================================================
# EXPORTACIÓN DE METADATA DEL MODELO
# =============================================================================
acc_final = resultados['Ensemble RF+XGB']['test_accuracy'].mean()
f1_final = resultados['Ensemble RF+XGB']['test_f1'].mean()
auc_final = resultados['Ensemble RF+XGB']['test_roc_auc'].mean()

metadata = {
    'nombre_modelo': 'CVEFixes_VulnerabilityClassifier_v1',
    'version': '1.0.0',
    'fecha_entrenamiento': datetime.now().isoformat(),
    'dataset': 'CVEFixes (simulado)',
    'tipo_modelo': 'Ensemble (RandomForest + XGBoost, VotingSoft)',
    'metricas_validacion_cruzada': {
        'cv_folds': 5,
        'accuracy_mean': round(float(acc_final), 4),
        'f1_mean': round(float(f1_final), 4),
        'roc_auc_mean': round(float(auc_final), 4),
        'umbral_requerido': 0.82,
        'cumple_umbral': bool(acc_final >= 0.82)
    },
    'features': {
        'tfidf_features': int(X_tfidf.shape[1]),
        'structural_features': int(X_struct.shape[1]),
        'selected_features': 3000,
        'balanceo': 'SMOTE(k_neighbors=5)'
    },
    'artefactos': list(rutas_exportadas.keys()),
    'lista_features_estructurales': list(df_features.columns),
    'instrucciones_inferencia': (
        'Para predecir: (1) extraer_features_codigo(codigo) → dict, '
        '(2) scaler.transform(dict_to_array), '
        '(3) vectorizador.transform([codigo]), '
        '(4) hstack([tfidf, struct_scaled]), '
        '(5) selector.transform(X_combined), '
        '(6) modelo_ensemble.predict(X_sel)'
    )
}

metadata_path = os.path.join(artifacts_dir, 'model_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'\n  📋 metadata.json → {os.path.getsize(metadata_path) / 1024:.1f} KB')

print('\n' + '=' * 60)
print('🎯 RESUMEN FINAL DE EXPORTACIÓN')
print('=' * 60)
print(f'  Directorio: {os.path.abspath(artifacts_dir)}')
print(f'  Accuracy CV (5-fold): {acc_final:.4f} ({"✅" if acc_final >= 0.82 else "❌"})')
print(f'  F1-Score CV:          {f1_final:.4f}')
print(f'  AUC-ROC CV:           {auc_final:.4f}')
print(f'  Umbral requerido:     0.8200')
print(f'  Estado:               {"✅ CUMPLE RÚBRICA" if acc_final >= 0.82 else "❌ NO CUMPLE"}')
print()
print('📂 Archivos exportados:')
for f_name in os.listdir(artifacts_dir):
    full_path = os.path.join(artifacts_dir, f_name)
    print(f'   ├─ {f_name} ({os.path.getsize(full_path)/1024:.1f} KB)')

---
## 🔄 BONUS: Función de Inferencia Rápida

Función de ejemplo para usar el modelo exportado en un contexto de producción.

In [ ]:
def predecir_vulnerabilidad(codigo_fuente, modelo, vectorizador, selector, scaler):
    """
    Función de inferencia rápida para clasificar un fragmento de código.
    
    Args:
        codigo_fuente (str): Código fuente a clasificar
        modelo: Clasificador entrenado
        vectorizador: TfidfVectorizer ajustado
        selector: SelectKBest ajustado
        scaler: MinMaxScaler ajustado
    
    Returns:
        dict: {'prediccion': 0/1, 'etiqueta': str, 'probabilidad_vuln': float, 'features_criticas': dict}
    """
    # 1. Extraer features estructurales
    features_dict = extraer_features_codigo(codigo_fuente)
    features_arr = np.array(list(features_dict.values())).reshape(1, -1)
    features_scaled = scaler.transform(features_arr.astype(float))
    X_struct_inf = csr_matrix(features_scaled)
    
    # 2. Vectorizar texto
    X_tfidf_inf = vectorizador.transform([codigo_fuente])
    
    # 3. Combinar y seleccionar features
    X_combined_inf = hstack([X_tfidf_inf, X_struct_inf])
    X_sel_inf = selector.transform(X_combined_inf)
    
    # 4. Predicción
    prediccion = modelo.predict(X_sel_inf)[0]
    probabilidades = modelo.predict_proba(X_sel_inf)[0]
    
    return {
        'prediccion': int(prediccion),
        'etiqueta': '⚠️  VULNERABLE' if prediccion == 1 else '✅ SEGURO',
        'probabilidad_vulnerable': float(probabilidades[1]),
        'probabilidad_seguro': float(probabilidades[0]),
        'features_criticas': {
            'danger_score': features_dict['total_danger_score'],
            'safety_score': features_dict['total_safety_score'],
            'ast_depth': features_dict['ast_depth'],
            'dangerous_calls': features_dict['dangerous_func_calls'],
            'sql_raw': features_dict['sql_raw_concat'],
            'subprocess_shell': features_dict['subprocess_shell'],
            'sanitization': features_dict['sanitization_present'],
        }
    }


# =============================================================================
# DEMOSTRACIÓN DE INFERENCIA
# =============================================================================
print('=' * 60)
print('🚀 DEMOSTRACIÓN DE INFERENCIA')
print('=' * 60)

casos_prueba = [
    {
        'nombre': 'SQL Injection (VULNERABLE)',
        'codigo': """
def buscar_usuario(username):
    sql = "SELECT * FROM users WHERE name='" + username + "'"
    return db.execute(sql)
"""
    },
    {
        'nombre': 'SQL Parameterizado (SEGURO)',
        'codigo': """
def buscar_usuario(username):
    sql = "SELECT * FROM users WHERE name=?"
    return db.execute(sql, (username,))
"""
    },
    {
        'nombre': 'Command Injection (VULNERABLE)',
        'codigo': """
import subprocess
def ejecutar(cmd_usuario):
    return subprocess.call('ls ' + cmd_usuario, shell=True)
"""
    },
    {
        'nombre': 'Subprocess Seguro (SEGURO)',
        'codigo': """
import subprocess, shlex
def listar_directorio(path):
    safe_path = shlex.quote(path)
    return subprocess.run(['ls', '-la', safe_path], capture_output=True, text=True).stdout
"""
    },
]

for caso in casos_prueba:
    resultado = predecir_vulnerabilidad(
        caso['codigo'],
        ensemble_clf, vectorizer, selector, scaler
    )
    print(f'\n📝 Caso: {caso["nombre"]}')
    print(f'   Predicción: {resultado["etiqueta"]}')
    print(f'   P(Vulnerable): {resultado["probabilidad_vulnerable"]:.4f}')
    print(f'   P(Seguro):     {resultado["probabilidad_seguro"]:.4f}')
    print(f'   Danger Score:  {resultado["features_criticas"]["danger_score"]}')
    print(f'   Safety Score:  {resultado["features_criticas"]["safety_score"]}')